In [ ]:
import pandas as pd
from IPython.display import display

# ===== 직접 지정 =====
model = "TimeXer"
dataset_name = "ETTh1"
seq_len = 96

# 설정
csv_path_base = f"/data/pcw_workspace/Time-Series-Library/loss_results/{model}/{dataset_name}/test_{seq_len}_summary.csv"
csv_path = f"/data/pcw_workspace/Time-Series-Library/loss_results/{model}/{dataset_name}_posiemb/test_{seq_len}_summary.csv"
pred_lens = [96, 192, 336, 720]

variant_order = [
    "base",
    "embedding",
    "projection",
    "embedding+projection\n(shared)",
    "embedding+projection\n(separate)"
]

# variant_order = [
#     "base",
#     "embedding",
#     "projection",
#     "embedding+projection"
# ]

# posiemb
def extract_variant(model_id):
    model_id = model_id.lower()
    if "_share" in model_id:
        return "embedding+projection\n(shared)"
    elif "_posiemb_emb" in model_id and "_posiemb_proj" in model_id:
            return "embedding+projection\n(separate)"
    elif "_posiemb_emb" in model_id:
        return "embedding"
    elif "_posiemb_proj" in model_id:
        return "projection"
    else:
        return "base"

# posienc
# def extract_variant(model_id):
#     model_id = model_id.lower()
#     if "_posienc_emb" in model_id and "_posienc_proj" in model_id:
#             return "embedding+projection"
#     elif "_posienc_emb" in model_id:
#         return "embedding"
#     elif "_posienc_proj" in model_id:
#         return "projection"
#     else:
#         return "base"

# 데이터 로드
df = pd.read_csv(csv_path)
df_base = pd.read_csv(csv_path_base)
df = pd.concat([df_base, df], ignore_index=True)
df = df[df["features_type"] == "M"].copy()
df["variant"] = df["model_id"].apply(extract_variant)
df["pred_len"] = pd.to_numeric(df["pred_len"], errors="coerce")

# groupby 평균 계산
avg = df.groupby(["variant", "pred_len"])[["mse", "mae"]].mean().reset_index()


# pivot 및 variant 순서 고정
pivot_mse = avg.pivot(index="pred_len", columns="variant", values="mse").reindex(columns=variant_order)
pivot_mae = avg.pivot(index="pred_len", columns="variant", values="mae").reindex(columns=variant_order)

# MultiIndex 컬럼 할당
pivot_mse.columns = pd.MultiIndex.from_tuples([(v, "MSE") for v in pivot_mse.columns])
pivot_mae.columns = pd.MultiIndex.from_tuples([(v, "MAE") for v in pivot_mae.columns])

# 병합 및 정렬 (열 순서 명시)
wide_df = pd.concat([pivot_mse, pivot_mae], axis=1)

# 열 순서 확정 (정렬이 아니라 명시!)
wide_df = wide_df[[ (v, m) for v in variant_order for m in ["MSE", "MAE"] ]]

# 인덱스 초기화 및 반올림
wide_df = wide_df.round(3).reset_index()


# features_type 컬럼 추가
wide_df.insert(0, (f"{model}", "features_type"), "M")

# 컬럼 MultiIndex 재설정
wide_df.columns = pd.MultiIndex.from_tuples(
    [(f"{model}", "features_type"), (f"{dataset_name}", "pred_len")] +
    [(v, m) for v in variant_order for m in ["MSE", "MAE"]]
)


# 강조 함수: 최솟값 bold, 차작값 underline
def highlight_min_second(row):
    styles = [""] * len(row)
    metrics = ["MSE", "MAE"]
    for metric in metrics:
        values = []
        col_indices = []
        for v in variant_order:
            col = (v, metric)
            if col in row.index:
                val = row[col]
                if pd.notna(val):
                    values.append(val)
                    col_indices.append(row.index.get_loc(col))
        if len(values) < 2:
            continue
        sorted_vals = sorted(set(values))
        min_val = sorted_vals[0]
        second_val = sorted_vals[1] if len(sorted_vals) > 1 else None
        for idx, val in zip(col_indices, values):
            if val == min_val:
                styles[idx] = "font-weight: bold"
            elif val == second_val:
                styles[idx] = "text-decoration: underline"
    return styles

# td_classes 설정
td_classes = pd.DataFrame("", index=wide_df.index, columns=wide_df.columns)

# 세로선: pred_len → base, base → emb, emb → proj, proj → emb+proj
td_classes[(f"{dataset_name}", "pred_len")] = "col-border"
td_classes[("base", "MSE")] = "col-border"
for v in variant_order[1:]:
    td_classes[(v, "MSE")] = "col-border"

# "features_type" 열에서 첫 행만 "M", 나머지는 빈 문자열 처리
feature_col = (f"{model}", "features_type")
wide_df[feature_col] = ["M"] + [""] * (len(wide_df) - 1)

# 스타일링
styled = (
    wide_df.style
    .apply(highlight_min_second, axis=1)
    .set_td_classes(td_classes)
    .format(precision=3)
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th.col0.level0", "props": [("background-color", "#e0e0e0")]},
        {"selector": "th.col1.level0", "props": [("background-color", "#e0e0e0")]},
        {"selector": "th", "props": [
            ("text-align", "center"),
            ("font-size", "14px"),
            ("font-weight", "bold"),
            ("background-color", "white"),
            ("color", "black"),
            ("border", "none")
        ]},
        {"selector": "td", "props": [
            ("text-align", "center"),
            ("font-size", "13px"),
            ("color", "black"),
            ("background-color", "white"),
            ("width", "80px"),
            ("border", "none")
        ]},
        {"selector": ".col-border", "props": [("border-left", "1px solid black")]}

    ])
)

# 출력
display(styled)

<h1> weight result (parameter search) </h1>

base 기준 성능 좋으면 빨강, 낮으면 파랑
loss sort했을때 상하위 25% 볼드 처리

In [91]:
import pandas as pd
from IPython.display import display
import os

# ===== 직접 지정 =====
model = "TimeXer"
dataset_name = "ETTh1"
seq_len = 96

# ===== 설정 =====
csv_path_base = f"/data/pcw_workspace/Time-Series-Library/loss_results/{model}/{dataset_name}/test_{seq_len}_summary.csv"
csv_path_embed = f"/data/pcw_workspace/Time-Series-Library/loss_results/{model}/{dataset_name}_posiemb/test_{seq_len}_summary.csv"
csv_path_weight = f"/data/pcw_workspace/Time-Series-Library/loss_results/{model}/{dataset_name}_posiemb_weight/test_{seq_len}_summary.csv"
pred_lens = [96, 192, 336, 720]

activation_order = [
    "base",
    "variate embedding",
    "variate embedding+\nscaling factor",
    "variate embedding+\nscaling factor(tanh)",
    "variate embedding+\nscaling factor(relu)",
    "variate embedding+\nscaling factor(sigmoid)"
]
model_id_col = "model_id"

def extract_activation_mode(model_id):
    if "weight" in model_id:
        if "sigmoid" in model_id:
            return "variate embedding+\nscaling factor(sigmoid)"
        elif "relu" in model_id:
            return "variate embedding+\nscaling factor(relu)"
        elif "tanh" in model_id:
            return "variate embedding+\nscaling factor(tanh)"
        else:
            return "variate embedding+\nscaling factor"
    elif "posiemb_emb" in model_id and not "posiemb_proj" in model_id:
        return "variate embedding"
    elif not "posiemb_proj" in model_id:
        return "base"

csv_paths = [csv_path_base, csv_path_embed, csv_path_weight]

dfs = []
for path in csv_paths:
    if os.path.exists(path):
        dfs.append(pd.read_csv(path))
    else:
        print(f"⚠️ 파일이 없습니다: {path}")

if dfs:
    df = pd.concat(dfs, ignore_index=True)

df["activation_mode"] = df[model_id_col].apply(extract_activation_mode)
df["dataset_name"] = dataset_name
df["pred_len"] = pd.to_numeric(df["pred_len"], errors="coerce")
df["weight"] = df[model_id_col].str.extract(r'weight_(\d*\.\d+)')

mask_no_weight = df["activation_mode"].isin(["base", "variate embedding"])
df.loc[mask_no_weight, "weight"] = "X"

# ---------- 스타일 함수 ----------
def color_by_base_comparison(df_block):
    styles = pd.DataFrame("", index=df_block.index, columns=df_block.columns)

    # 전체 mse, mae 값 수집 (숫자인 것만)
    all_mse_vals, all_mae_vals = [], []
    for col in df_block.columns:
        if isinstance(col, tuple) and col[1] == "mse":
            all_mse_vals.extend(pd.to_numeric(df_block[col], errors="coerce").dropna().tolist())
        elif isinstance(col, tuple) and col[1] == "mae":
            all_mae_vals.extend(pd.to_numeric(df_block[col], errors="coerce").dropna().tolist())

    if not all_mse_vals or not all_mae_vals:
        return styles

    mse_low, mse_high = pd.Series(all_mse_vals).quantile([0.25, 0.75])
    mae_low, mae_high = pd.Series(all_mae_vals).quantile([0.25, 0.75])

    try:
        base_mse_col = ("base", "mse")
        base_mae_col = ("base", "mae")
        base_mse_vals = pd.to_numeric(df_block[base_mse_col], errors="coerce")
        base_mae_vals = pd.to_numeric(df_block[base_mae_col], errors="coerce")
        base_mse = base_mse_vals[base_mse_vals.notna()].iloc[0]
        base_mae = base_mae_vals[base_mae_vals.notna()].iloc[0]
    except Exception:
        base_mse, base_mae = None, None

    for col in df_block.columns:
        if not (isinstance(col, tuple) and col[1] in ("mse", "mae")):
            continue

        metric = col[1]
        base_val = base_mse if metric == "mse" else base_mae
        low, high = (mse_low, mse_high) if metric == "mse" else (mae_low, mae_high)

        for i, val in df_block[col].items():
            try:
                num_val = float(val)
            except:
                continue

            # 상/하위 25%이면 볼드
            if num_val <= low or num_val >= high:
                styles.at[i, col] += "font-weight: bold;"

            # base 컬럼은 항상 볼드 + 검정
            if col[0] == "base":
                styles.at[i, col] += "font-weight: bold; color: black !important;"

            # base 대비 색상
            if base_val is not None:
                if num_val < base_val:
                    styles.at[i, col] += "color: red !important;"
                elif num_val > base_val:
                    styles.at[i, col] += "color: blue !important;"

    return styles

def merge_like_first_col(x):
    """첫 컬럼 병합 효과(보이는 셀 하나만 두고 나머지는 투명 처리) + 짝수행일 때 위칸에 배치"""
    styles = pd.DataFrame("", index=x.index, columns=x.columns)
    n = len(x)
    mid = (n - 1) // 2  # 짝수면 위칸(예: 2행이면 0), 홀수면 가운데

    for i in range(n):
        if i != mid:
            styles.iloc[i, 0] += "color: transparent;"

    styles.iloc[mid, 0] += "font-weight: bold;"

    # 짝수행이면 위칸에 여백을 줘서 '세로 중앙'처럼 보이게
    if n % 2 == 0:
        styles.iloc[mid, 0] += "padding-bottom: 10px;"

    # 첫 컬럼 텍스트 정렬 중앙
    styles.iloc[:, 0] += "text-align: center;"

    return styles

def set_last_row_border(x):
    styles = pd.DataFrame("", index=x.index, columns=x.columns)
    styles.iloc[-1, :] = "border-top: 3px solid black"
    return styles

# ===== 표 생성 =====
for pred_len in pred_lens:
    df_pred = df[(df["pred_len"] == pred_len) & (df["activation_mode"].isin(activation_order))]

    pivot_df = df_pred.pivot_table(index=["weight"], columns=["activation_mode"], values=["mse", "mae"])
    pivot_df.columns = pivot_df.columns.swaplevel(0, 1)

    # 빈 점('·')만 있는 activation은 제거
    activation_to_keep = []
    for act in activation_order:
        mse_col = (act, "mse")
        mae_col = (act, "mae")
        if mse_col in pivot_df.columns and mae_col in pivot_df.columns:
            all_dots = all(
                str(row[mse_col]) == "·" and str(row[mae_col]) == "·"
                for _, row in pivot_df.iterrows()
            )
            if not all_dots:
                activation_to_keep.append(act)

    new_columns = [(act, metric) for act in activation_to_keep for metric in ["mse", "mae"]]
    pivot_df = pivot_df[new_columns]
    pivot_df.columns.names = ["activation_mode", "metric"]

    pivot_df = pivot_df.round(3).reset_index()

    # weight 정렬 ('X'는 뒤로)
    def sort_key(x):
        try:
            return float(x)
        except:
            return float('-inf')

    pivot_df = pivot_df.sort_values(by="weight", key=lambda x: x.map(sort_key)).reset_index(drop=True)
    pivot_df.insert(0, "No", range(1, len(pivot_df) + 1))
    pivot_df = pivot_df.fillna("·")

    # === 헤더 재조립 ===
    new_cols = []
    for c in pivot_df.columns:
        if isinstance(c, tuple):
            top, bot = c[0], (c[1] if len(c) > 1 else "")
            new_cols.append((top, bot))
        else:
            if c == "No":
                new_cols.append(("No", ""))
            elif c == "weight":
                # 표시 라벨만 scaling factor로 (표시용이니 top은 공란, bottom에 텍스트)
                new_cols.append(("", "scaling factor"))
            else:
                new_cols.append((str(c), ""))

    pivot_df.columns = pd.MultiIndex.from_tuples(new_cols, names=["activation_mode", "metric"])
    cols = list(pivot_df.columns)

    # 첫 번째 컬럼은 bottom 레벨 'pred'
    cols[0] = ("", "pred")
    # 두 번째 컬럼은 bottom 레벨 'scaling factor'가 되도록 보정
    if not (isinstance(cols[1], tuple) and cols[1][1] == "scaling factor"):
        cols[1] = ("", "scaling factor")

    pivot_df.columns = pd.MultiIndex.from_tuples(cols, names=pivot_df.columns.names)

    # 첫 컬럼(pred)에 pred_len 표시 (짝수행이면 위칸)
    n_rows = len(pivot_df)
    first_col = pivot_df.columns[0]  # ("", "pred")
    pivot_df[first_col] = ""
    mid_row = (n_rows - 1) // 2
    pivot_df.iloc[mid_row, 0] = str(pred_len)

    # ===== 경계선 =====
    # 1) 각 구획의 시작 위치(세로선 넣을 열) 계산
    flat_cols = list(pivot_df.columns)
    pos_by_col = {c: i for i, c in enumerate(flat_cols)}

    # (a) pred | scaling factor 사이 선
    scaling_pos = next(i for i, c in enumerate(flat_cols)
                    if isinstance(c, tuple) and c[1] == "scaling factor")

    # (b) 각 activation 블록 시작(mse) 위치들
    group_start_positions = []
    for act in activation_to_keep:  # base 포함해서 각 블록 시작에 선
        key = (act, "mse")
        if key in pos_by_col:
            group_start_positions.append(pos_by_col[key])

    # 최종 세로선 위치들
    separator_positions = [scaling_pos] + group_start_positions

    # 2) 바디(td)에 세로선
    def set_custom_borders(x):
        styles = pd.DataFrame("", index=x.index, columns=x.columns)
        for pos in separator_positions:
            styles.iloc[:, pos] += "border-left: 1.5px solid black;"
        return styles

    # 3) 헤더(th)에도 같은 세로선 (MultiIndex 2줄 모두 반영)
    header_sep_styles = []
    for pos in separator_positions:
        nth = pos + 1  # CSS nth-child는 1-base
        header_sep_styles += [
            {"selector": f"thead th:nth-child({nth})",
            "props": [("border-left", "1.5px solid black")]},
            {"selector": f"thead th.col_heading.level0.col{pos}",
            "props": [("border-left", "1.5px solid black")]},
            {"selector": f"thead th.col_heading.level1.col{pos}",
            "props": [("border-left", "1.5px solid black")]},
        ]

    styled = (
        pivot_df.style
        .format(precision=3)
        .apply(merge_like_first_col, axis=None)
        .apply(set_custom_borders, axis=None)     # ← 바디 세로선
        .apply(color_by_base_comparison, axis=None)
        .set_table_styles(
            [
                {"selector": "th", "props": [
                    ("text-align", "center"),
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("padding", "6px"),
                    ("background-color", "white"),
                    ("color", "#333"),
                    ("white-space", "pre-line"),
                    ("border-bottom", "2.5px solid black"),  # 헤더 하단 가로선
                ]},
                {"selector": "td", "props": [
                    ("text-align", "center"),
                    ("font-size", "13px"),
                    ("padding", "4px 8px"),
                    ("background-color", "white"),
                    ("color", "#222"),
                ]},
                {"selector": "caption", "props": [
                    ("caption-side", "top"),
                    ("background-color", "white"),
                    ("color", "black"),
                    ("font-size", "16px"),
                    ("font-weight", "bold"),
                    ("padding", "10px"),
                    ("text-align", "center"),
                    ("border-bottom", "2.5px solid black"),
                ]},
            ] + header_sep_styles,                 # ← 헤더 세로선
            overwrite=False
        )
        .set_caption(f"{model} | {dataset_name}")
        .hide_index()
    )

    print(f"📊 pred_len = {pred_len}")
    display(styled)

📊 pred_len = 96


/tmp/ipykernel_378228/2598008459.py:256: FutureWarning: this method is deprecated in favour of `Styler.hide(axis='index')`
  pivot_df.style


📊 pred_len = 192


/tmp/ipykernel_378228/2598008459.py:256: FutureWarning: this method is deprecated in favour of `Styler.hide(axis='index')`
  pivot_df.style


📊 pred_len = 336


/tmp/ipykernel_378228/2598008459.py:256: FutureWarning: this method is deprecated in favour of `Styler.hide(axis='index')`
  pivot_df.style


📊 pred_len = 720


/tmp/ipykernel_378228/2598008459.py:256: FutureWarning: this method is deprecated in favour of `Styler.hide(axis='index')`
  pivot_df.style
